In [1]:
import numpy.random as rand
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge as ridge
from sklearn.linear_model import Lasso as lasso
from sklearn.linear_model import LinearRegression as ols


from sklearn.ensemble import RandomForestRegressor as rfr
from sklearn.tree import DecisionTreeRegressor as reg_tree
from sklearn.ensemble import AdaBoostRegressor as ada_reg
from sklearn.ensemble import GradientBoostingRegressor as gbr
from sklearn.metrics import mean_squared_error as mse
from sklearn.model_selection import train_test_split
import copy

import matplotlib
from sklearn.metrics import mean_squared_error as mse

In [13]:
import rpy2.robjects as ro
readRDS = ro.r['readRDS']

path = 'C:/integraion_RCT_and_Obs/integration_RCT_and_Obs/02_build/data/ACTG.obj'
data =  readRDS(path) 
train_data = data[2]
test_data = data[1]

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

Y = np.array(train_data[1])
Z = np.array(train_data[2])
cd80 = np.array(train_data[3])
cd40 = np.array(train_data[4])
ID = np.array(train_data[5])

X = np.array([cd80, cd40]).T

Y_sd = np.std(Y)
Y_mean = np.mean(Y)

X = scaler.fit_transform(X)
Y = (Y - Y_mean)/Y_sd

In [16]:
# sepalate  RCT and Obs
X_O = X[ID == "O"]
X_E = X[ID == "R"]
Y_O = Y[ID == "O"]
Y_E = Y[ID == "R"]
T_O = Z[ID == "O"]
T_E = Z[ID == "R"]

In [17]:
regs = [rfr(n_estimators=i) for i in [10, 20, 40, 60, 100, 150, 200]]
regs += [reg_tree(max_depth=i) for i in [5, 10, 20, 30, 40, 50]]
regs += [ada_reg(n_estimators=i) for i in [10, 20, 50, 70, 100, 150, 200]]
regs += [gbr(n_estimators=i) for i in [50, 70, 100, 150, 200]]

In [18]:
def get_best_for_data(X, Y, regs):
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2) # doesn't change X
    val_errs = []
    models = []
    for reg in regs:
        model = copy.deepcopy(reg)
        model.fit(x_train, y_train)
        val_errs.append(mse(y_test, model.predict(x_test)))
        models.append(copy.deepcopy(model))
    min_ind = val_errs.index(min(val_errs))
    print(str(model)[:40], val_errs[min_ind])
    return copy.deepcopy(models[min_ind])

In [31]:
f1pred_exp = get_best_for_data(X_E[T_E>0].reshape(-1,2), Y_E[T_E>0], regs)
f0pred_exp = get_best_for_data(X_E[T_E==0].reshape(-1,2), Y_E[T_E==0], regs)
f1pred_obs = get_best_for_data(X_O[T_O>0].reshape(-1,2), Y_O[T_O>0], regs)
f0pred_obs = get_best_for_data(X_O[T_O==0].reshape(-1,2), Y_O[T_O==0], regs)

GradientBoostingRegressor(n_estimators=2 0.1545290431244941
GradientBoostingRegressor(n_estimators=2 0.35976963453352095
GradientBoostingRegressor(n_estimators=2 0.5063169778758968
GradientBoostingRegressor(n_estimators=2 0.9784798888666346


In [32]:
omega = f1pred_obs.predict(X_E.reshape(-1,2)) - f0pred_obs.predict(X_E.reshape(-1,2)) 
tau = f1pred_exp.predict(X_E.reshape(-1,2)) - f0pred_exp.predict(X_E.reshape(-1,2))
print(omega.shape, tau.shape)

(20,) (20,)


In [33]:
omega_O = f1pred_obs.predict(X_O.reshape(-1,2)) - f0pred_obs.predict(X_O.reshape(-1,2)) 
tau_O = f1pred_exp.predict(X_O.reshape(-1,2)) - f0pred_exp.predict(X_O.reshape(-1,2))
print(omega.shape, tau.shape)

(20,) (20,)


In [34]:
eta_est = tau - omega
print(eta_est.shape)
eta_est_O = tau_O - omega_O
print(eta_est_O.shape)

# eta_ridge= ridge()
eta_ridge = get_best_for_data(X_E.reshape(-1, 2), eta_est, [ridge(alpha=a) for a in [1e-10]])# , 1e-5, 1e-4]])# ,1e-2,1,1e+2,1e+4, 1e+5]])
print(mse(eta_est, eta_ridge.predict(X_E.reshape(-1, 2))))
# print(mse(eta, -6*kappa*X_full))

(20,)
(200,)
Ridge(alpha=1e-10) 3.382384480427193
1.2294095287517353


In [35]:
eta_ridge = get_best_for_data(X_E.reshape(-1, 2), eta_est, [ridge(alpha=a) for a in [1e-10]])# , 1e-5, 1e-4]])# ,1e-2,1,1e+2,1e+4, 1e+5]])
print(mse(eta_est, eta_ridge.predict(X_E.reshape(-1, 2))))

Ridge(alpha=1e-10) 0.19941506910961104
1.1700822186911009


In [36]:
test_X = np.array([np.array(test_data[3]),np.array(test_data[4])]).T
test_X = scaler.transform(test_X)

In [37]:
omega =  f1pred_obs.predict(test_X) - f0pred_obs.predict(test_X)
eta   = eta_ridge.predict(test_X)

In [47]:
pred = (omega + eta) * Y_sd

In [48]:
np.mean(pred)

424.66648130722507

In [53]:
np.savetxt('Kallus_pred.csv', pred, delimiter=',')